# qPCR analysis pipeline by Alan Henrique - rev. 30/10/2026

#### 1. Load libraries and import data
#### 2. Keep relevant columns
<pre>   Minimum required columns for automated processing include `Sample`, `Target` and `Cq`, but can also include `Cq_mean` at this points</pre>
#### 3. Calculate mean and standard deviation, in case these were not calculated before
#### 4. (OPTIONAL) For samples with standard deviation between replicates above 0.5, remove the sample with the highest z-score and recalculate the mean and Stdev after removal
#### 5. Group replicates (keeping only Cq_mean and Cq_std for each sample)
#### 6. Add a column with Cq values for the housekeeping gene
#### 7. Calculate ΔCq
7.1. In case you are using the ΔΔCq methos, calculate the average of the control group, paying attention to the correct grouping depending on your independent variables

7.2. Next, calculate ΔΔCq
#### 8. Export processed data to a .csv file

### 1. Import libraries and data
Use the function `read_csv` or `read_excel` according to the extension of the data file you're using

In [ ]:
import pandas as pd
import numpy as np

# Add the filepath to your data file
filepath = r'C:/Users/...'

# df = pd.read_csv(filepath, sep=',', header=0, index_col=0) # Use this if you're reading data from a .csv file

# OR

# df = pd.read_excel(filepath, header=0, index_col=0, sheet_name=0) # Use this if you're reading data from a .xlsx file

### In case you have more than one file:

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

path = r"C:\Users\..." # This is the path to your folder containing the data files

all_files = glob.glob(path + "/*.csv") # Grabs all file names that end in .csv (remember to change to .xlsx if you're using Excel files)
li = []

print(all_files) # Check that you're getting the right files, feel free to comment this entire line

for file in all_files:
    filename=os.path.basename(file)
    df = pd.read_csv(file, index_col=None, header=24) # Use this if you're reading data from a .csv file
    # df = pd.read_excel(file, index_col=None, header=24) # Use this if you're reading data from a .xlsx file
    print(filename) # Checking that you're reading the correct file
    li.append(df)

df = pd.concat(li, axis=0, ignore_index=True) # This concatenates all data into a single dataframe

['C:\\Users\\alanm\\OneDrive\\Área de Trabalho\\teste pcr\\tables\\GAPDH_results.csv', 'C:\\Users\\alanm\\OneDrive\\Área de Trabalho\\teste pcr\\tables\\IL-1b_results.csv']
GAPDH_results.csv
IL-1b_results.csv


#### 2. Keeping only the relevant columns and loading a .csv table containing the identification for each sample
<pre>  You have to at least have the columns `Sample`, `Target` and `Cq`, but you can also include `Cq_mean` and `Cq_std`
  It is convenient to rename the columns to the standard strings used throughout the rest of the code</pre>

In [ ]:
# Keeping only relevant columns. Pay attention: depending on your qPCR platform, you might need to change the column names.
cols_to_keep = ['Sample','Cq', 'Target']
df = df[cols_to_keep]

# Reading a .csv file that contains information on samples, such as age, treatment condition, sex etc.
samples = pd.read_csv('samples.csv')

# Renaming columns to standard names used throughout the code. Make sure that your data file has the same column names; otherwise, change them here.
rename_dict = {
    'Sample': 'sample',
    'Cq': 'cq',
    'Target': 'target',
    'Cq_mean': 'cq_mean',
    'Cq_std': 'cq_std'
}

# Renaming columns according to the above dictionary
df = df.rename(columns=rename_dict)

# Merging samples dataframe with the qPCR dataframe, so that all relevant information is available in `df`
df = pd.merge(df, samples, how='left', on=['sample'])

#### 3. Calculate mean and standard deviation, in case it hasn't been done already
##### <pre>    To avoid errors, we first convert Cq values to numerical and remove NaN entries. Make sure to check your data before and after.</pre>

In [ ]:
# Converting cq values to numeric and removing NAs
df['cq'] = pd.to_numeric(df['cq'], errors='coerce')
df = df[df['cq'].notna()]

# Calculating mean and standard deviation for each replicate. Remember to adjust the grouping variables in case your data has multiple grouping variables (such as age, treatment etc).
stats = df.groupby(['sample', 'target'])['cq'].agg(cq_mean='mean', cq_std='std').reset_index()

# Merging with the original dataframe
df = df.merge(stats, on=['sample', 'target'], how='left')

#### 4. (OPTIONAL) For samples with standard deviation between replicates above 0.5, remove the sample with the highest z-score and recalculate mean and SD afterwards.

In [ ]:
def filter_replicates_qpcr(df, group_cols=['target', 'sample'], std_limit=0.5):
    """
    Removes the replicate that is farthest from the mean only if the STD of the group
    is higher than the std_limit variable, and only if there are more than 2 replicates.
    """
    
    idxs_to_remove = []
    grouping_cols = df.groupby(group_cols)

    print(f"--- Starting cleanup (Limit STD: {std_limit}) ---")

    for name, group in grouping_cols:
        # Calculates the current STD of the group
        current_std = group['cq'].std()
        
        # CONDITION: only acts if STD is above the limit and if there are more than 2 replicates
        if current_std > std_limit:
            if len(group) > 2:
                # Fetches the replicate that is farthest from the mean
                mean_original = group['cq'].mean()
                distances = (group['cq'] - mean_original).abs()
                idx_outlier = distances.idxmax()
                
                idxs_to_remove.append(idx_outlier)
                print(f"Grupo {name}: STD {current_std:.3f} > {std_limit}. Removing replicate {df.loc[idx_outlier, 'well']}.")
            else:
                print(f"Grupo {name}: STD {current_std:.3f} above {std_limit}, but fewer than 3 replicates. Not removing replicate.")

    # Criar o DF limpo
    df_clean = df.drop(idxs_to_remove).copy()
    
    # Recalcular as colunas de média e desvio para os valores que sobraram
    df_clean['cq_mean'] = df_clean.groupby(group_cols)['cq'].transform('mean')
    df_clean['cq_std'] = df_clean.groupby(group_cols)['cq'].transform('std')

    print(f"\nTotal replicates cleaned: {len(idxs_to_remove)}")
    return df_clean

# Como usar:
df = filter_replicates_qpcr(df, group_cols=['target', 'sample'], std_limit=0.5)

--- Starting cleanup (Limit STD: 0.5) ---
Grupo ('GAPDH', 'Sample 9'): STD 1.224 above 0.5, but fewer than 3 replicates. Not removing replicate.
Grupo ('IL-10', 'Sample 1'): STD 0.583 above 0.5, but fewer than 3 replicates. Not removing replicate.
Grupo ('IL-10', 'Sample 11'): STD 1.298 above 0.5, but fewer than 3 replicates. Not removing replicate.
Grupo ('IL-10', 'Sample 13'): STD 0.570 above 0.5, but fewer than 3 replicates. Not removing replicate.
Grupo ('IL-10', 'Sample 17'): STD 0.673 above 0.5, but fewer than 3 replicates. Not removing replicate.
Grupo ('IL-10', 'Sample 18'): STD 0.919 above 0.5, but fewer than 3 replicates. Not removing replicate.
Grupo ('IL-10', 'Sample 19'): STD 0.747 above 0.5, but fewer than 3 replicates. Not removing replicate.
Grupo ('IL-10', 'Sample 20'): STD 0.954 above 0.5, but fewer than 3 replicates. Not removing replicate.
Grupo ('IL-10', 'Sample 5'): STD 1.315 above 0.5, but fewer than 3 replicates. Not removing replicate.
Grupo ('IL-10', 'Sample 7

#### 5. Grouping replicates (keeping only cq_mean and cq_std values)

In [ ]:
def drop_replicates(df, group_cols=['target', 'sample']):
    '''
    Remove as replicatas pra ficar apenas com o valor médio para cada amostra
    '''
    
    df.drop(['cq'], axis=1, inplace=True)
    df.drop_duplicates(inplace=True)
    #df = df[df['cq_mean'] < 35] # In case you want to remove samples with very high Cq, which indicate a very low gene expression
    return df

# Como usar:
df = drop_replicates(df)

#### 6. Adding column with housekeeping gene Cq and calculating ΔCq

In [ ]:
def normalize_by_housekeeping(df, ref_gene='GAPDH', sample_col='sample', target_col='target', mean_col='cq_mean'):
    """
    Creates cq_ref column and removes the housekeeping gene from the target column. Calculates ΔCq afterwards.
    """
    
    # 1. Map Cq mean of the housekeeping gene for each sample
    # We only use values where 'target' is your selected housekeeping gene
    ref_values = df[df[target_col] == ref_gene].groupby(sample_col)[mean_col].first()
    
    # 2. Creates cq_ref column in the original dataframe using this map
    df['cq_ref'] = df[sample_col].map(ref_values)
    
    # Verification: checking if all samples have cq information for the housekeeping gene
    samples_without_hk = df[df['cq_ref'].isna()][sample_col].unique()
    if len(samples_without_hk) > 0:
        print(f"Warning: Samples {samples_without_hk} don't have {ref_gene} data!")

    # 3. Removing housekeeping gene entries
    df_result = df[df[target_col] != ref_gene].copy()
    
    # 4. Calculating ΔCq (gene of interest - housekeeping)
    df_result['delta_cq'] = df_result[mean_col] - df_result['cq_ref']
    
    print(f"Done. {ref_gene} removed and ΔCq successfully calculated.")
    return df_result

df = normalize_by_housekeeping(df, ref_gene='YOUR HOUSEKEEPING GENE')

Done. GAPDH removed and ΔCq successfully calculated.


#### 7. Calculate relative gene expression (ΔΔCq)
##### <pre>    7.1. In case you are using the ΔΔCq method, calculate the mean of the control group (always paying attention to group correctly considering the independent variables)</pre>
##### <pre>    7.2. Afterwards, calculate ΔΔCq</pre>

In [28]:
def calculate_livak(df, control_cond='DMSO', target_col='target', delta_col='delta_cq'):
    """
    Calculates ΔΔCq and fold change (2^-ddCq) off of a control group.
    """
    
    # 1. Define what samples are in the control group
    control_mask = (df['condition'] == control_cond)
    
    # 2. Calculate ΔCq mean by TARGET within the control group
    # This generates an individual baseline for each gene
    control_means = df[control_mask].groupby(target_col)[delta_col].mean()
    
    # Verify if all targets have information on the control group
    targets_without_control = set(df[target_col].unique()) - set(control_means.index)
    if targets_without_control:
        print(f"Warning: Genes {targets_without_control} are not in the control group!")

    # 3. Map control means back into the original dataframe
    # Creates a temporary column to make sure the subtraction is being done with the correct control and gene
    df['mean_delta_cq_control'] = df[target_col].map(control_means)
    
    # 4. Calculating ΔΔCq
    # ddCq = dCq(sample) - Média_dCq(control)
    df['deltadelta_cq'] = df[delta_col] - df['mean_delta_cq_control']
    
    # 5. Calculating fold change (2^-ddCq)
    # We use the negative signal because Cq is inversely proportional to the amount of cDNA
    df['2^-ddcq'] = 2 ** (-df['deltadelta_cq'])
    
    # Optional: Remove the temporary mean column to clean the dataframe.
    # df = df.drop(columns=['mean_delta_cq_control'])
    
    print(f"ΔΔCq successfully calculated using {control_cond} as the control group.")
    return df

### Additional: calculate fold change using 2^-ΔCq

In [ ]:
# 1. Calcualte raw relative expression (2^-dCq) for all samples
# In this case, we use ΔCq instead of ΔΔCq
df['rel_exp_raw'] = 2 ** (-df['delta_cq'])

# 2. Define the control group reference
ref_mask = (df['treatment'] == 'UT')

# 3. Calculate mean raw expression for EACH GENE within the control group
# This generates a series where index is the gene and the value is the cq mean of the control group for that gene
means_per_target = df[ref_mask].groupby('target')['rel_exp_raw'].mean()

# 4. Map these means back into the original dataframe
# .map() guarantees that each line is divided by the mean corresponding to that specific gene
df['fold_change'] = df['rel_exp_raw'] / df['target'].map(means_per_target)

# --- Optional verification: checking that all means are equal to 1 ---
print("Checking means (they should all be 1.0):")
print(df[ref_mask].groupby('target')['fold_change'].mean())

# If any one gene does not have a reference group, it will be assigned NaN in fold_change.
# Let us check if it happened:
genes_without_ref = df[df['fold_change'].isna()]['target'].unique()
if len(genes_without_ref) > 0:
    print(f"\nWarning: Genes {genes_without_ref} do not have the reference group!")

Checking means (they should all be 1.0):
target
IL-10    1.0
IL-1b    1.0
IL-6     1.0
mKL 2    1.0
s-KL     1.0
Name: fold_change, dtype: float64



#### Exporting processed data to a .csv file for statistical analysis (which can be performed in a different software).

In [ ]:
df.to_csv('FILE NAME HERE.csv', sep=',', index=False)